# VisDSR: saved recovery run

**Replacement collection: 192 responses, the same frozen development protocol.**
The earlier interactive run completed, but its response files were not retained. This notebook records a replacement run using the verified GPU fix. It does not recreate the lost answers or change tasks, prompts, model revisions, or generation settings.

1. Enable **Internet** and **GPU T4 ×2**.
2. Choose the top-right **Save Version → Save & Run All**. This notebook intentionally blocks interactive execution.
3. Open the **new saved version's Log**. Check for **SAVED RUN MODE: Batch**, **GPU MASK CHECK: 2 CUDA devices tested**, and **STARTING model1 / model2**. The older failed Version 1 is separate.
4. After the saved version finishes, open its **Output** and download **visdsr_improvements_complete.zip**. Completion requires **192/192**; a partial result uses **visdsr_improvements_backup.zip**.

The notebook pins source commit `9b4d02b`, which includes the CUDA device-selection fix. Both models run on separate GPUs with one load each. There is no notebook time cutoff. Kaggle session limits or infrastructure failures can still interrupt collection; a local ZIP is durable only after Kaggle retains it in a saved version or you download it.

No previous ZIP is needed for this replacement collection. Compatible backups can be attached before starting a saved run. New responses are assessed without selecting or retrying answers based on correctness. See the [recovery instructions](https://github.com/Abhishekgit01/VisDSR/blob/main/docs/IMPROVEMENT_RECOVERY.md).


In [ ]:
import os

run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
print("SAVED RUN MODE:", run_type or "unknown", flush=True)
if run_type != "Batch":
    raise RuntimeError(
        "Use the top-right Save Version > Save & Run All. "
        "This recovery notebook blocks interactive collection so that responses "
        "can be retained in the new saved version's Output."
    )

SOURCE_COMMIT = "9b4d02bb189b66daf2ec99790270f0f481301247"
PARALLEL_MODELS = True             # One model per T4; single GPU falls back to sequential
BACKUP_INPUT = "auto"              # Or an explicit diagnostic/improvement ZIP or extracted dataset
if type(PARALLEL_MODELS) is not bool:
    raise ValueError("PARALLEL_MODELS must be True or False.")
print("192 planned responses; baseline requests reused; no repeated calls for cached answers.")


## Setup, validation, and optional restoration

This checks the original frozen inputs and the new intervention freeze. It compiles each task grammar with both actual tokenizers, without downloading model weights. Compatible diagnostic backups from the previous 96-call notebook can be reused. Completed main-study ZIPs belong to a different experiment.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
print("SAVED RUN MODE:", run_type or "unknown", flush=True)
if run_type != "Batch":
    raise RuntimeError("Use the top-right Save Version > Save & Run All to run this entire notebook.")
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Import this notebook into Kaggle and enable T4 GPUs and Internet.")
if "BACKUP_INPUT" not in globals() or "SOURCE_COMMIT" not in globals():
    raise RuntimeError(
        "Configuration did not finish. Use the top-right Save Version > Save & Run All. "
        "If this is already a saved run, check its Log for the first error in Configuration."
    )
PROJECT = Path("/kaggle/working/VisDSR_improvements_recovery")
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--no-checkout",
                    "https://github.com/Abhishekgit01/VisDSR.git", str(PROJECT)], check=True)
    subprocess.run(["git", "fetch", "--depth", "1", "origin", SOURCE_COMMIT], cwd=PROJECT, check=True)
    subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=PROJECT, check=True)
actual_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT,
                               check=True, capture_output=True, text=True).stdout.strip()
if actual_commit != SOURCE_COMMIT:
    raise RuntimeError("Existing checkout has a different source revision; preserve its backups and use a fresh notebook.")
print("VERIFIED SOURCE COMMIT:", actual_commit, flush=True)
if not (PROJECT / "interventions/FREEZE.json").is_file():
    raise RuntimeError("This checkout predates the improvement comparison; import the published notebook.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
                "-r", str(PROJECT / "environment/requirements.txt"),
                "-r", str(PROJECT / "environment/requirements-inference.txt"),
                "-r", str(PROJECT / "interventions/requirements.txt")], check=True)
subprocess.run(["make", "build"], cwd=PROJECT, check=True)
for command in (("diagnostics", "check"), ("interventions", "check"), ("interventions", "decoder-check")):
    subprocess.run([sys.executable, "-m", *command], cwd=PROJECT, check=True)

if BACKUP_INPUT == "auto":
    candidates = sorted(Path("/kaggle/input").rglob("*diagnostics*.zip"))
    candidates += sorted(Path("/kaggle/input").rglob("*improvements*.zip"))
    for manifest in sorted(Path("/kaggle/input").rglob("MANIFEST.json")):
        info = json.loads(manifest.read_text())
        if info.get("protocol_id") in ("visdsr-dsu-diagnostics-20261007", "visdsr-dsu-interventions-20261007"):
            candidates.append(manifest.parent)
else:
    candidates = [Path(BACKUP_INPUT)]
for backup in candidates:
    if backup.is_dir():
        info = json.loads((backup / "MANIFEST.json").read_text())
    else:
        import zipfile
        with zipfile.ZipFile(backup) as archive:
            info = json.loads(archive.read("MANIFEST.json"))
    module = "interventions" if info["protocol_id"] == "visdsr-dsu-interventions-20261007" else "diagnostics"
    subprocess.run([sys.executable, "-m", module, "restore", "--input", str(backup)], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "interventions", "score"], cwd=PROJECT, check=True)
print("Setup complete. Compatible cached answers will be reused.")


## Collect in the saved version

Both checkpoints run automatically in **Save & Run All**. An interactive attempt stops before the collection subprocess starts. Every completed answer updates its raw cache and model backup; the parent combines both afterward.

Check this saved version's Log for **FINAL STATUS: complete; 192/192 saved**. Once the saved run is complete, its Output is the download location. A completed draft cell by itself does not confirm that Kaggle retained the files.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
print("SAVED RUN MODE:", run_type or "unknown", flush=True)
if run_type != "Batch":
    raise RuntimeError(
        "Use the top-right Save Version > Save & Run All. "
        "This recovery notebook blocks interactive collection so that responses "
        "can be retained in the new saved version's Output."
    )

project = Path("/kaggle/working/VisDSR_improvements_recovery")
if "PARALLEL_MODELS" not in globals():
    raise RuntimeError(
        "Configuration did not finish. Use the top-right Save Version > Save & Run All. "
        "If this is already a saved run, check its Log for the first error in Configuration."
    )
command = [sys.executable, "-u", "-m", "interventions", "overnight", "--output-dir", "/kaggle/working"]
if not PARALLEL_MODELS:
    command.append("--sequential")
subprocess.run(command, cwd=project, check=True)


## Export and mark the replacement run

This final cell marks the run as replacement collection after loss of the earlier draft's raw files, then refreshes both exports. The note is included in the archive's `run_status.json`; it does not alter response caches, scoring, or the frozen protocol.

Download **visdsr_improvements_complete.zip** from this saved version's Output at **192/192**. For partial collection, attach the downloaded **visdsr_improvements_backup.zip** before starting another saved run. No discarded answers are reconstructed from the earlier progress log.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_improvements_recovery")
results = project / "results/followup/visdsr-dsu-interventions-20261007"
status_path = results / "run_status.json"
if not status_path.is_file():
    raise RuntimeError(
        "No collection status is available to export. "
        "Use the top-right Save Version > Save & Run All to run this entire notebook. "
        "If that saved run failed, check its Log for the first error before trying Export."
    )
status = json.loads(status_path.read_text())
status["recovery"] = {
    "reason": "Replacement collection after loss of an unsaved interactive session's response files",
    "earlier_draft_reported_responses": 192,
    "earlier_raw_files_available": False,
    "source_commit": SOURCE_COMMIT,
    "kaggle_run_type": run_type,
}
status_path.write_text(json.dumps(status, indent=2) + "\n")
output = Path("/kaggle/working")
backup = output / "visdsr_improvements_backup.zip"
subprocess.run([sys.executable, "-m", "interventions", "export", "--output", str(backup)], cwd=project, check=True)
summary = json.loads((results / "summary.json").read_text())
if summary["complete"]:
    complete = output / "visdsr_improvements_complete.zip"
    subprocess.run([sys.executable, "-m", "interventions", "export", "--output", str(complete)], cwd=project, check=True)
    print("COMPLETE ZIP:", complete, flush=True)
(output / "visdsr_improvements_run_status.json").write_bytes(status_path.read_bytes())
print(f"SAVED: {summary['saved']}/192; complete={summary['complete']}", flush=True)
print("Download from THIS saved version's Output:", backup, flush=True)
